# Models

In [0]:
pip install optuna xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
base_dir = "/Workspace/Users/jonc@bocdaz.ca/Var_imp/var_imp_final_test/"

## Random Forest

In [0]:
import optuna
from optuna.integration import XGBoostPruningCallback
from optuna.visualization import plot_optimization_history

from sklearn.ensemble import RandomForestRegressor as RF
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, StratifiedKFold

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
import xgboost as xgb
from xgboost.callback import EarlyStopping
import warnings

In [0]:
def stratified_by_group_kfold_cv_mse_rf(
    trial,
    params,
    X,
    y,
    groups,
    splits,
):
    # Enforce fast settings
    params = params.copy()
    params.setdefault("bootstrap", True)
    params.setdefault("n_jobs", 1)

    mses = []

    for fold_id, (train_idx, test_idx) in enumerate(splits):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test,  y_test  = X[test_idx],  y[test_idx]

        model = RF(
            **params,
            random_state=42 + fold_id,
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_train, y_train)

        preds = model.predict(X_test)
        mse = np.sqrt(mean_squared_error(y_test, preds))
        mses.append(mse)

        # ----------------------
        # Optuna pruning
        # ----------------------
        trial.report(mse, step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(mses))


# ============================================================
# Optuna objective (same name)
# ============================================================

def objective_rf(trial, X, y, groups, splits):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 250),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0), #0.3?
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
    }

    return stratified_by_group_kfold_cv_mse_rf(
        trial, params, X, y, groups, splits
    )


# ============================================================
# Optuna tuning wrapper (same name)
# ============================================================

def tune_rf_optuna(
    X, y, groups,
    corr, sd, stu, sch,
    n_trials=50,
    n_jobs=-1,
):
    # Convert ONCE
    X = X.values if hasattr(X, "values") else np.asarray(X)
    y = y.values if hasattr(y, "values") else np.asarray(y)
    groups = groups.values if hasattr(groups, "values") else np.asarray(groups)

    # --------------------------
    # Precompute CV splits ONCE
    # --------------------------
    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42,
    )
    splits = list(skf.split(X, groups))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=10,
            n_warmup_steps=2,
        ),
    )

    study.optimize(
        lambda trial: objective_rf(
            trial, X, y, groups, splits
        ),
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=True,
    )

    fig = plot_optimization_history(study)
    fig.write_html(
        f"{base_dir}rf_history_{corr}_{sd}_{stu}_{sch}.html"
    )

    return study.best_params


## MERF

In [0]:
class MERF_CPU:
    def __init__(self, rf_params=None, max_iter=6, tol=1e-4, random_state=42):
        rf_params = rf_params or {}
        rf_params.setdefault("n_estimators", 200)
        rf_params.setdefault("max_depth", 12)
        rf_params.setdefault("random_state", random_state)
        rf_params.setdefault("n_jobs", 1)      # critical for EM loop
        rf_params.setdefault("bootstrap", True)
        rf_params.setdefault("warm_start", True)

        self.rf_params = rf_params
        self.max_iter = max_iter
        self.tol = tol

        self.rf = RF(**self.rf_params)
        self.gamma = None
        self.group_map = None
        self.sigma2_hat = None

    # --------------------------------------------------
    # Fit
    # --------------------------------------------------

    def fit(self, X, y, groups):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        groups = np.asarray(groups)

        # Encode groups
        unique_groups, group_idx = np.unique(groups, return_inverse=True)
        counts = np.bincount(group_idx).astype(np.float32)

        # Init
        gamma = np.zeros(len(unique_groups), dtype=np.float32)
        mu_hat = np.zeros_like(y)
        sigma2 = np.var(y)

        n_trees_base = self.rf.n_estimators

        for it in range(self.max_iter):
            # -----------------------------
            # Fixed effects (warm start)
            # -----------------------------
            self.rf.n_estimators = n_trees_base * (it + 1)
            self.rf.fit(X, y - gamma[group_idx])

            mu_hat[:] = self.rf.predict(X)

            # -----------------------------
            # Random intercept update
            # -----------------------------
            resid = y - mu_hat
            gamma_new = np.bincount(group_idx, weights=resid) / counts

            # -----------------------------
            # Variance + convergence
            # -----------------------------
            resid_full = y - (mu_hat + gamma_new[group_idx])
            sigma2_new = np.mean(resid_full * resid_full)

            if abs(sigma2_new - sigma2) < self.tol * (sigma2 + 1e-12):
                gamma = gamma_new
                sigma2 = sigma2_new
                break

            gamma = gamma_new
            sigma2 = sigma2_new

        self.gamma = gamma
        self.group_map = dict(zip(unique_groups, np.arange(len(unique_groups))))
        self.sigma2_hat = sigma2

    # --------------------------------------------------
    # Predict (FAST)
    # --------------------------------------------------

    def predict(self, X, groups):
        X = np.asarray(X, dtype=np.float32)
        groups = np.asarray(groups)

        y_fe = self.rf.predict(X)

        idx = np.array(
            [self.group_map.get(g, -1) for g in groups],
            dtype=int,
        )

        re = np.zeros(len(groups), dtype=np.float32)
        mask = idx >= 0
        re[mask] = self.gamma[idx[mask]]

        return y_fe + re


In [0]:
def stratified_by_group_kfold_merf_cpu_loss(
    trial,
    MERFClass,
    rf_params,
    max_iter,
    tol,
    X,
    y,
    groups,
    splits,
):
    mses = []

    for fold_id, (train_idx, test_idx) in enumerate(splits):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        groups_train = groups[train_idx]
        groups_test = groups[test_idx]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            merf = MERFClass(
                rf_params=rf_params,
                max_iter=max_iter,
                tol=tol,
                random_state=42 + fold_id,
            )
            merf.fit(X_train, y_train, groups_train)
            preds = merf.predict(X_test, groups_test)

        mse = np.sqrt(mean_squared_error(y_test, preds))
        mses.append(mse)

        # --------------------
        # Optuna pruning
        # --------------------
        trial.report(mse, step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(mses))



def objective_cpu_merf(trial, X, y, groups, splits):
    rf_params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 250),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0), # 0.3?
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "bootstrap": True,
        "n_jobs": 1,
    }

    # EM should converge quickly
    max_iter = trial.suggest_int("max_iter", 3, 7)
    tol = trial.suggest_float("tol", 1e-5, 1e-3, log=True)

    return stratified_by_group_kfold_merf_cpu_loss(
        trial,
        MERF_CPU,
        rf_params,
        max_iter,
        tol,
        X,
        y,
        groups,
        splits,
    )

def tune_merf_cpu(
    X, y, groups,
    corr, sd, stu, sch,
    n_trials=50,
    n_jobs=-1,
):
    X_np = np.asarray(X)
    y_np = np.asarray(y)
    groups_np = np.asarray(groups)

    # --------------------------
    # Precompute CV splits ONCE
    # --------------------------
    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42,
    )
    splits = list(skf.split(X_np, groups_np))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=10,
            n_warmup_steps=2,
        ),
    )

    study.optimize(
        lambda trial: objective_cpu_merf(
            trial, X_np, y_np, groups_np, splits
        ),
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=True,
    )

    fig = plot_optimization_history(study)
    fig.write_html(
        f"{base_dir}merf_history_{corr}_{sd}_{stu}_{sch}.html"
    )

    return study.best_params.copy()

## XGBoost

In [0]:
def stratified_by_group_kfold_cv_mse_xgb_cpu(
    trial, params, X, y, groups, splits
):
    mse_list = []

    for fold_id, (train_idx, test_idx) in enumerate(splits):
        # --------------------------
        # Train / test split
        # --------------------------
        X_train, y_train = X[train_idx], y[train_idx]
        X_test,  y_test  = X[test_idx],  y[test_idx]

        # Inner validation split (for early stopping)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train,
            test_size=0.2,
            random_state=42
        )

        # --------------------------
        # DMatrix construction
        # --------------------------
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval   = xgb.DMatrix(X_val, label=y_val)
        dtest  = xgb.DMatrix(X_test)

        xgb_params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "subsample": params["subsample"],
            "colsample_bytree": params["colsample_bytree"],
            "min_child_weight": params["min_child_weight"],
            "gamma": params["gamma"],
            "nthread": 1,          # avoid oversubscription
            "seed": 42 + fold_id,
        }

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            booster = xgb.train(
                xgb_params,
                dtrain,
                num_boost_round=1000,          # high ceiling
                evals=[(dval, "val")],
                early_stopping_rounds=50,
                verbose_eval=False,
            )

        preds = booster.predict(dtest)
        mse = np.sqrt(mean_squared_error(y_test, preds))
        mse_list.append(mse)

        # --------------------------
        # Optuna pruning
        # --------------------------
        trial.report(mse, step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(mse_list))


# ============================================================
# Optuna objective (UNCHANGED NAME)
# ============================================================

def objective_xgb_cpu(trial, X, y, groups, splits):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
    }

    return stratified_by_group_kfold_cv_mse_xgb_cpu(
        trial, params, X, y, groups, splits
    )


# ============================================================
# Tuning wrapper (UNCHANGED NAME)
# ============================================================

def tune_xgb_cpu(
    X, y, groups,
    corr, sd, stu, sch,
    n_trials=100,
    n_jobs=-1
):
    # Convert once
    X_np = np.asarray(X)
    y_np = np.asarray(y)
    groups_np = np.asarray(groups)

    # --------------------------
    # Precompute CV splits
    # --------------------------
    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )
    splits = list(skf.split(X_np, groups_np))

    # --------------------------
    # Optuna study
    # --------------------------
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=10,
            n_warmup_steps=2
        ),
    )

    study.optimize(
        lambda trial: objective_xgb_cpu(
            trial, X_np, y_np, groups_np, splits
        ),
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=True,
    )

    fig = plot_optimization_history(study)
    fig.write_html(
        f"{base_dir}xgb_history_{corr}_{sd}_{stu}_{sch}.html"
    )

    return study.best_params.copy()

## MEGB

In [0]:
class MEGB_CPU:
    """
    Fast Mixed Effects Gradient Boosting
    (XGBoost fixed effects + random intercepts)
    """

    def __init__(
        self,
        xgb_params=None,
        max_iter=6,          # MUCH smaller is enough
        tol=1e-4,
        random_state=42
    ):
        xgb_params = xgb_params or {}
        xgb_params.setdefault("learning_rate", 0.05)
        xgb_params.setdefault("max_depth", 6)
        xgb_params.setdefault("subsample", 0.9)
        xgb_params.setdefault("colsample_bytree", 0.9)
        xgb_params.setdefault("objective", "reg:squarederror")
        xgb_params.setdefault("tree_method", "hist")
        xgb_params.setdefault("n_jobs", 1)
        xgb_params.setdefault("verbosity", 0)
        xgb_params.setdefault("seed", random_state)

        self.xgb_params = xgb_params
        self.max_iter = max_iter
        self.tol = tol

        self.booster = None
        self.gamma = None
        self.group_map = None
        self.sigma2_hat = None

    # --------------------------------------------------
    # Fit
    # --------------------------------------------------

    def fit(self, X, y, groups):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        groups = np.asarray(groups)

        # Encode groups
        unique_groups, group_idx = np.unique(groups, return_inverse=True)
        counts = np.bincount(group_idx).astype(np.float32)

        # Validation split (ONCE)
        tr_idx, val_idx = train_test_split(
            np.arange(len(y)), test_size=0.2, random_state=42
        )

        dtrain = xgb.DMatrix(X[tr_idx], label=y[tr_idx])
        dvalid = xgb.DMatrix(X[val_idx], label=y[val_idx])
        dfull  = xgb.DMatrix(X)

        # Init
        gamma = np.zeros(len(unique_groups), dtype=np.float32)
        mu_hat = np.zeros_like(y)
        sigma2 = np.var(y)

        booster = None

        # EM loop
        for _ in range(self.max_iter):
            # -----------------------------
            # Fixed effects (warm start)
            # -----------------------------
            dtrain.set_label(y[tr_idx] - gamma[group_idx[tr_idx]])
            dvalid.set_label(y[val_idx] - gamma[group_idx[val_idx]])

            booster = xgb.train(
                self.xgb_params,
                dtrain,
                num_boost_round=500,
                evals=[(dvalid, "val")],
                early_stopping_rounds=20,
                xgb_model=booster,
                verbose_eval=False,
            )

            mu_hat[:] = booster.predict(dfull)

            # -----------------------------
            # Random intercept update
            # -----------------------------
            resid = y - mu_hat
            gamma_new = np.bincount(group_idx, weights=resid) / counts

            # -----------------------------
            # Variance update + convergence
            # -----------------------------
            resid_full = y - (mu_hat + gamma_new[group_idx])
            sigma2_new = np.mean(resid_full ** 2)

            if abs(sigma2_new - sigma2) < self.tol * (sigma2 + 1e-12):
                gamma = gamma_new
                sigma2 = sigma2_new
                break

            gamma = gamma_new
            sigma2 = sigma2_new

        # Save
        self.booster = booster
        self.gamma = gamma
        self.group_map = dict(zip(unique_groups, np.arange(len(unique_groups))))
        self.sigma2_hat = sigma2

    # --------------------------------------------------
    # Predict (FAST, vectorized)
    # --------------------------------------------------

    def predict(self, X, groups):
        X = np.asarray(X, dtype=np.float32)
        groups = np.asarray(groups)

        dmat = xgb.DMatrix(X)
        y_fe = self.booster.predict(dmat)

        group_idx = np.array(
            [self.group_map.get(g, -1) for g in groups],
            dtype=int
        )
        re = np.zeros(len(groups), dtype=np.float32)
        mask = group_idx >= 0
        re[mask] = self.gamma[group_idx[mask]]

        return y_fe + re


In [0]:
def stratified_by_group_kfold_megb_loss_cpu(
    trial,
    MEGBClass,
    xgb_params,
    max_iter,
    tol,
    X,
    y,
    groups,
    splits,
):
    mse_list = []

    for fold_id, (train_idx, test_idx) in enumerate(splits):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        groups_tr = groups[train_idx]
        groups_te = groups[test_idx]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            model = MEGBClass(
                xgb_params=xgb_params,
                max_iter=max_iter,
                tol=tol,
                random_state=42 + fold_id,
            )
            model.fit(X_tr, y_tr, groups_tr)
            preds = model.predict(X_te, groups_te)

        mse = np.sqrt(mean_squared_error(y_te, preds))
        mse_list.append(mse)

        # ----------------------
        # Optuna pruning
        # ----------------------
        trial.report(mse, step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(mse_list))



def objective_megb_cpu(trial, X, y, groups, splits):
    xgb_params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
    }

    # EM should NOT be tuned aggressively
    max_iter = trial.suggest_int("max_iter", 3, 8)
    tol = trial.suggest_float("tol", 1e-5, 1e-3, log=True)

    return stratified_by_group_kfold_megb_loss_cpu(
        trial,
        MEGB_CPU,
        xgb_params,
        max_iter,
        tol,
        X,
        y,
        groups,
        splits,
    )


def tune_megb_cpu(
    X, y, groups,
    corr, sd, stu, sch,
    n_trials=50,
    n_jobs=-1
):
    X_np = np.asarray(X)
    y_np = np.asarray(y)
    groups_np = np.asarray(groups)

    # --------------------------
    # Precompute CV splits ONCE
    # --------------------------
    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )
    splits = list(skf.split(X_np, groups_np))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=10,
            n_warmup_steps=2
        ),
    )

    study.optimize(
        lambda trial: objective_megb_cpu(
            trial, X_np, y_np, groups_np, splits
        ),
        n_trials=n_trials,
        n_jobs=n_jobs,
        show_progress_bar=True,
    )

    fig = plot_optimization_history(study)
    fig.write_html(
        f"{base_dir}"
        f"megb_history_{corr}_{sd}_{stu}_{sch}.html"
    )

    return study.best_params